In [ ]:
RUN_K3D = True


In [ ]:
# %% Cell 0: Setup and Imports

RUN_K3D = True

import os
import sys
import numpy as np
import matplotlib.pyplot as plt # For colormaps in K3D if needed
import h5py
from tqdm.notebook import tqdm
from pyquaternion import Quaternion
import logging
from typing import Dict, List, Optional, Tuple, Any
import k3d
import json # For loading config from HDF5 if stored as string

# --- Add Project Root to sys.path ---
NOTEBOOK_DIR = os.path.abspath('') # Assumes notebook is in 'notebooks' or similar
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# --- Import Custom Modules ---
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import Box as NuScenesDataClassesBox

from src.config_loader import MDetectorConfigAccessor
from src.core.constants import OcclusionResult, POINT_LABEL_DTYPE
from src.data_utils.nuscenes_helper import get_scene_sweep_data_sequence, get_lidar_sweep_data
from src.utils.transformations import transform_points_numpy
from src.utils.validation_utils import get_gt_dynamic_points_for_sweep
from src.utils.notebook_helpers import create_k3d_plot_with_points # General K3D helper
from src.visualization.k3d_visualizer import _get_data_from_hdf5_for_sweep # HDF5 helper
from src.data_utils.label_generation import find_instances_in_scene, get_interpolated_extrapolated_boxes_for_instance

# For BEV Animation (if you want to reuse parts or generate it here)
from src.visualization.frame_composers import compose_gt_vs_mdet_frame
from src.visualization.video_generator import generate_video_from_hdf5_list

print("Cell 0: Imports and Paths - OK")

In [ ]:
# %% Cell 1: Configuration, NuScenes Init, Paths, Scene/Sweep Selection

# --- Configure Logging (Optional, for less verbose output in this notebook) ---
logging.basicConfig(level=logging.WARNING) # Set to WARNING or ERROR for cleaner output
# logging.getLogger('src.core.m_detector.base').setLevel(logging.INFO)
# logging.getLogger('src.data_utils.nuscenes_helper').setLevel(logging.INFO)
print("Logging configured (set to WARNING).")

# --- Load Configuration using Accessor ---
config_accessor: Optional[MDetectorConfigAccessor] = None
try:
    config_file_path = os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml')
    config_accessor = MDetectorConfigAccessor(config_file_path)
    print(f"Configuration loaded successfully from: {config_file_path}")
except Exception as e:
    print(f"Error loading config: {e}")
    # Potentially raise e or exit if config is critical

# --- Initialize NuScenes ---
nusc: Optional[NuScenes] = None
if config_accessor:
    nuscenes_params = config_accessor.get_nuscenes_params()
    try:
        nusc = NuScenes(
            version=nuscenes_params.get('version', 'v1.0-mini'),
            dataroot=nuscenes_params.get('dataroot', '/data/sets/nuscenes'),
            verbose=nuscenes_params.get('verbose_init', False)
        )
        print(f"NuScenes SDK initialized (Version: {nuscenes_params.get('version')}).")
    except Exception as e:
        print(f"Error initializing NuScenes SDK: {e}")
else:
    print("ERROR: MDetectorConfigAccessor failed. Cannot initialize NuScenes SDK.")

# --- Define Paths for GT Labels and M-Detector Outputs (HDF5) ---
GT_LABELS_HDF5_DIR: Optional[str] = None
MDET_RESULTS_HDF5_DIR: Optional[str] = None # This will be the specific experiment run
if config_accessor:
    nuscenes_params = config_accessor.get_nuscenes_params()
    GT_LABELS_HDF5_DIR = nuscenes_params.get('label_path')
    if GT_LABELS_HDF5_DIR and not os.path.isabs(GT_LABELS_HDF5_DIR):
        GT_LABELS_HDF5_DIR = os.path.join(PROJECT_ROOT, GT_LABELS_HDF5_DIR)

    # IMPORTANT: Point this to the specific experiment output directory you want to visualize
    # This might be manually set or derived from a naming convention
    mdet_output_paths = config_accessor.get_mdetector_output_paths()
    # Example: if your script saved to 'output/mdetector_results/experiment_run_XYZ'
    # You might need to adjust this path based on how your 'run_mdetector_and_save.py' names output folders.
    # For now, let's assume it's the general save_path from config, but you'll likely make this more specific.
    EXPERIMENT_NAME_TO_VISUALIZE = "" # CHANGE THIS to your actual experiment folder name
    MDET_RESULTS_HDF5_DIR = os.path.join(mdet_output_paths.get('save_path', 'output/mdetector_results'), EXPERIMENT_NAME_TO_VISUALIZE)
    if not os.path.isabs(MDET_RESULTS_HDF5_DIR):
        MDET_RESULTS_HDF5_DIR = os.path.join(PROJECT_ROOT, MDET_RESULTS_HDF5_DIR)

print(f"GT Labels HDF5 Directory: {GT_LABELS_HDF5_DIR if GT_LABELS_HDF5_DIR else 'Not configured'}")
print(f"M-Detector Results HDF5 Directory: {MDET_RESULTS_HDF5_DIR if MDET_RESULTS_HDF5_DIR else 'Not configured'}")

# --- Scene and Sweep Selection ---
# You can make these configurable or hardcode for a specific visualization
TARGET_SCENE_NAME_VIZ = 'scene-0103'
TARGET_SWEEP_INDEX_IN_SCENE_VIZ = 50 # Example: Visualize the 51st sweep (0-indexed)

target_scene_rec_viz: Optional[Dict[str, Any]] = None
TARGET_LIDAR_SD_TOKEN_VIZ: Optional[str] = None
target_sweep_data_dict_viz: Optional[Dict[str, Any]] = None

if nusc and config_accessor:
    scene_found = False
    for scene in nusc.scene:
        if scene['name'] == TARGET_SCENE_NAME_VIZ:
            target_scene_rec_viz = scene
            scene_found = True
            break
    if not scene_found:
        print(f"ERROR: Scene '{TARGET_SCENE_NAME_VIZ}' not found.")
    else:
        print(f"Target Scene for Visualization: '{target_scene_rec_viz['name']}' (Token: {target_scene_rec_viz['token']})")
        all_sweeps_for_scene_viz = list(get_scene_sweep_data_sequence(nusc, target_scene_rec_viz['token']))
        if 0 <= TARGET_SWEEP_INDEX_IN_SCENE_VIZ < len(all_sweeps_for_scene_viz):
            target_sweep_data_dict_viz = all_sweeps_for_scene_viz[TARGET_SWEEP_INDEX_IN_SCENE_VIZ]
            TARGET_LIDAR_SD_TOKEN_VIZ = target_sweep_data_dict_viz['lidar_sd_token']
            print(f"  Selected Target Sweep Index: {TARGET_SWEEP_INDEX_IN_SCENE_VIZ}, SD Token: ...{TARGET_LIDAR_SD_TOKEN_VIZ[-8:]}")
        else:
            print(f"ERROR: Target sweep index {TARGET_SWEEP_INDEX_IN_SCENE_VIZ} is out of bounds for scene '{TARGET_SCENE_NAME_VIZ}' (has {len(all_sweeps_for_scene_viz)} sweeps).")
else:
    print("Skipping Scene/Sweep Selection: NuScenes SDK or Config Accessor not available.")

print("\nCell 1: Setup and Configuration - OK")


In [ ]:
# %% Cell 2: Load M-Detector Results and GT for Target Sweep

mdet_predictions_structured: Optional[np.ndarray] = None
gt_indices_dict_viz: Optional[Dict[str, Any]] = None
raw_points_global_viz: Optional[np.ndarray] = None # All points from the original sweep
mdet_points_global_viz: Optional[np.ndarray] = None # Points from MDet HDF5 (should match raw after filtering)
mdet_labels_numeric_viz: Optional[np.ndarray] = None

if nusc and target_scene_rec_viz and TARGET_LIDAR_SD_TOKEN_VIZ and \
   MDET_RESULTS_HDF5_DIR and GT_LABELS_HDF5_DIR and config_accessor:

    # 1. Load M-Detector HDF5 results for the target sweep
    mdet_scene_hdf5_filename = f"mdet_results_{target_scene_rec_viz['name']}.h5"
    mdet_scene_hdf5_filepath = os.path.join(MDET_RESULTS_HDF5_DIR, mdet_scene_hdf5_filename)

    if os.path.exists(mdet_scene_hdf5_filepath):
        try:
            with h5py.File(mdet_scene_hdf5_filepath, 'r') as mdet_h5:
                mdet_predictions_structured = _get_data_from_hdf5_for_sweep(
                    mdet_h5, TARGET_LIDAR_SD_TOKEN_VIZ,
                    'sweep_lidar_sd_tokens', 'all_points_predictions', 'points_predictions_indices'
                )
            if mdet_predictions_structured is not None:
                print(f"M-Detector predictions loaded for sweep {TARGET_LIDAR_SD_TOKEN_VIZ[:8]}... ({mdet_predictions_structured.shape[0]} points)")
                mdet_points_global_viz = np.stack((mdet_predictions_structured['x'],
                                                   mdet_predictions_structured['y'],
                                                   mdet_predictions_structured['z']), axis=-1)
                mdet_labels_numeric_viz = mdet_predictions_structured['mdet_label'].astype(int)
            else:
                print(f"Warning: No M-Detector predictions found for sweep {TARGET_LIDAR_SD_TOKEN_VIZ} in {mdet_scene_hdf5_filepath}")
        except Exception as e:
            print(f"Error loading M-Detector HDF5 results: {e}")
    else:
        print(f"ERROR: M-Detector results HDF5 file not found: {mdet_scene_hdf5_filepath}")

    # 2. Get Raw LiDAR points for the target sweep (these are the points M-Detector *would have* processed)
    # This requires knowing the filtering M-Detector applied.
    # We'll load the config from the M-Detector HDF5 file to get the exact filtering used for *that run*.
    mdet_run_config_dict = None
    if os.path.exists(mdet_scene_hdf5_filepath):
        try:
            with h5py.File(mdet_scene_hdf5_filepath, 'r') as mdet_h5_cfg_load:
                if '_config_json_str' in mdet_h5_cfg_load:
                    config_str_data = mdet_h5_cfg_load['_config_json_str'][()]
                    config_str = config_str_data.decode('utf-8') if isinstance(config_str_data, bytes) else str(config_str_data)
                    mdet_run_config_dict = json.loads(config_str)
                    print("  Successfully loaded M-Detector run-specific config from its HDF5.")
        except Exception as e_cfg_hdf5:
            print(f"  Warning: Could not load run-specific config from MDet HDF5: {e_cfg_hdf5}")

    min_range_for_sweep = config_accessor.get_point_pre_filtering_params().get('min_range_meters', 1.0)
    max_range_for_sweep = config_accessor.get_point_pre_filtering_params().get('max_range_meters', 80.0)
    if mdet_run_config_dict: # Prioritize config from the actual run
        try:
            run_filtering_params = mdet_run_config_dict.get('m_detector', {}).get('point_pre_filtering', {})
            min_r_run = run_filtering_params.get('min_range_meters')
            max_r_run = run_filtering_params.get('max_range_meters')
            if min_r_run is not None: min_range_for_sweep = float(min_r_run)
            if max_r_run is not None: max_range_for_sweep = float(max_r_run)
            print(f"  Using MDet range from HDF5 config for point loading: {min_range_for_sweep:.1f}-{max_range_for_sweep:.1f}m")
        except Exception:
            print("  Could not parse range from HDF5 config, using current script's config for point loading.")


    if target_sweep_data_dict_viz:
        points_sf = target_sweep_data_dict_viz['points_sensor_frame']
        T_gl = target_sweep_data_dict_viz['T_global_lidar']
        if points_sf.shape[0] > 0:
            ranges_sf = np.linalg.norm(points_sf[:, :3], axis=1)
            range_mask_sf = (ranges_sf >= min_range_for_sweep) & (ranges_sf <= max_range_for_sweep)
            filtered_points_sf = points_sf[range_mask_sf]
            raw_points_global_viz = transform_points_numpy(filtered_points_sf, T_gl)
            print(f"Raw LiDAR points loaded and filtered for target sweep: {raw_points_global_viz.shape[0]} points")

            # Sanity check point counts
            if mdet_points_global_viz is not None and raw_points_global_viz.shape[0] != mdet_points_global_viz.shape[0]:
                print(f"WARNING: Point count mismatch! Raw filtered: {raw_points_global_viz.shape[0]}, MDet HDF5: {mdet_points_global_viz.shape[0]}")
                print("This can happen if the M-Detector run used different pre-filtering than assumed here.")
                print("For visualization, we will proceed using points from M-Detector's HDF5 if available, otherwise raw.")
                if mdet_points_global_viz.shape[0] == 0 and raw_points_global_viz.shape[0] > 0 :
                     print("MDet HDF5 has 0 points, but raw sweep has points. Check MDet processing for this sweep.")
                     # Potentially set mdet_labels_numeric_viz to empty if mdet_points_global_viz is empty
                     mdet_labels_numeric_viz = np.array([], dtype=int)
                elif mdet_points_global_viz.shape[0] > 0:
                     raw_points_global_viz = mdet_points_global_viz # Prefer MDet HDF5 points if counts differ but MDet has points
                     print("  Using points from M-Detector HDF5 for consistency in visualization.")
                # If mdet_points_global_viz is None but raw_points_global_viz exists, mdet_labels_numeric_viz will be None, handled later

        else:
            raw_points_global_viz = np.empty((0,3))
            print("Raw LiDAR sweep has no points.")
    else:
        print("Target sweep data dictionary not available.")


    # 3. Load GT labels for the target sweep, using the *raw_points_global_viz* as the reference point cloud
    #    and the M-Detector's filtering parameters for consistency.
    if raw_points_global_viz is not None and raw_points_global_viz.shape[0] > 0:
        gt_labels_scene_hdf5_filepath = os.path.join(GT_LABELS_HDF5_DIR, f"gt_point_labels_{target_scene_rec_viz['name']}.h5")
        validation_params = config_accessor.get_validation_params()
        velocity_threshold_gt_viz = validation_params.get('gt_velocity_threshold', 0.5)

        if os.path.exists(gt_labels_scene_hdf5_filepath):
            gt_indices_dict_viz = get_gt_dynamic_points_for_sweep(
                nusc=nusc,
                sweep_data_dict=target_sweep_data_dict_viz,
                all_points_global_mdetector=raw_points_global_viz, # Crucial: GT is aligned to these points
                gt_labels_scene_hdf5_path=gt_labels_scene_hdf5_filepath,
                velocity_threshold=velocity_threshold_gt_viz,
                mdetector_min_range=min_range_for_sweep, # Use the same range as M-Detector for GT filtering
                mdetector_max_range=max_range_for_sweep
            )
            if gt_indices_dict_viz and gt_indices_dict_viz.get('error_msg') is None:
                print(f"GT indices loaded for sweep. Dynamic: {len(gt_indices_dict_viz.get('gt_dynamic_indices',[]))}, Static: {len(gt_indices_dict_viz.get('gt_static_indices',[]))}")
            elif gt_indices_dict_viz:
                print(f"Error loading GT indices: {gt_indices_dict_viz.get('error_msg')}")
            else:
                print("Failed to get GT indices dictionary.")
        else:
            print(f"ERROR: GT HDF5 file not found for scene: {gt_labels_scene_hdf5_filepath}")
    elif raw_points_global_viz is not None and raw_points_global_viz.shape[0] == 0:
        print("No raw points to load GT against (raw_points_global_viz is empty).")
    else:
        print("Raw points for target sweep not loaded, cannot load GT labels.")
else:
    print("Skipping data loading: Prerequisites not met.")

print("\nCell 2: Data Loading - OK")

In [ ]:
# %% Cell 3: Data Preparation for K3D Visualization

k3d_points_to_plot: Dict[str, Tuple[np.ndarray, int, float]] = {}
gt_boxes_for_k3d: List[NuScenesDataClassesBox] = []

if raw_points_global_viz is not None and \
   mdet_labels_numeric_viz is not None and \
   gt_indices_dict_viz and gt_indices_dict_viz.get('error_msg') is None and \
   config_accessor:

    k3d_cfg = config_accessor.get_k3d_plot_params()
    num_total_points = raw_points_global_viz.shape[0]

    # 1. RANSAC Ground Points
    ransac_mask = (mdet_labels_numeric_viz == OcclusionResult.PRELABELED_STATIC_GROUND.value)
    ransac_pts = raw_points_global_viz[ransac_mask]
    if ransac_pts.shape[0] > 0:
        k3d_points_to_plot['RANSAC_Ground'] = (
            ransac_pts,
            k3d_cfg.get('ransac_ground_color_hex', 0x008000), # Dark Green
            k3d_cfg.get('point_size', 0.05) * k3d_cfg.get('ransac_ground_point_size_factor', 0.8)
        )
    print(f"RANSAC Ground points: {ransac_pts.shape[0]}")

    # 2. M-Detector Dynamic Predictions
    mdet_is_dynamic_mask = (mdet_labels_numeric_viz == OcclusionResult.OCCLUDING_IMAGE.value)

    # 3. Ground Truth Dynamic Status (relative to raw_points_global_viz)
    gt_is_dynamic_mask = np.zeros(num_total_points, dtype=bool)
    if gt_indices_dict_viz.get('gt_dynamic_indices', np.array([])).size > 0:
        gt_is_dynamic_mask[gt_indices_dict_viz['gt_dynamic_indices']] = True
    
    gt_is_static_labeled_mask = np.zeros(num_total_points, dtype=bool) # GT static (has instance)
    if gt_indices_dict_viz.get('gt_static_indices', np.array([])).size > 0:
        gt_is_static_labeled_mask[gt_indices_dict_viz['gt_static_indices']] = True
        
    gt_is_unlabeled_mask = np.zeros(num_total_points, dtype=bool) # GT unlabeled (background)
    if gt_indices_dict_viz.get('unlabeled_indices', np.array([])).size > 0:
        gt_is_unlabeled_mask[gt_indices_dict_viz['unlabeled_indices']] = True
        
    # Combine GT static and GT unlabeled into "not GT dynamic" for TP/FP/FN/TN calculation
    # Exclude RANSAC points from TP/FP/FN/TN if they are to be shown separately.
    # If RANSAC points should also be part of TN if they are GT_Static, adjust logic.
    # For now, RANSAC is a separate category.
    
    non_ransac_mask = ~ransac_mask

    # 4. TP, FP, FN, TN
    tp_mask = gt_is_dynamic_mask & mdet_is_dynamic_mask & non_ransac_mask
    tp_pts = raw_points_global_viz[tp_mask]
    if tp_pts.shape[0] > 0:
        k3d_points_to_plot['TP_Dynamic'] = (
            tp_pts,
            k3d_cfg.get('tp_color_hex', 0x00FF00), # Bright Green
            k3d_cfg.get('point_size', 0.05) * k3d_cfg.get('tp_point_size_factor', 1.2)
        )
    print(f"True Positives (Dynamic): {tp_pts.shape[0]}")

    fp_mask = ~gt_is_dynamic_mask & mdet_is_dynamic_mask & non_ransac_mask # MDet says dyn, GT says not dyn
    fp_pts = raw_points_global_viz[fp_mask]
    if fp_pts.shape[0] > 0:
        k3d_points_to_plot['FP_Dynamic'] = (
            fp_pts,
            k3d_cfg.get('fp_color_hex', 0xFF0000), # Red
            k3d_cfg.get('point_size', 0.05) * k3d_cfg.get('fp_point_size_factor', 1.2)
        )
    print(f"False Positives (Dynamic): {fp_pts.shape[0]}")

    fn_mask = gt_is_dynamic_mask & ~mdet_is_dynamic_mask & non_ransac_mask # MDet says not dyn, GT says dyn
    fn_pts = raw_points_global_viz[fn_mask]
    if fn_pts.shape[0] > 0:
        k3d_points_to_plot['FN_Dynamic'] = (
            fn_pts,
            k3d_cfg.get('fn_color_hex', 0xFF8C00), # DarkOrange
            k3d_cfg.get('point_size', 0.05) * k3d_cfg.get('fn_point_size_factor', 1.2)
        )
    print(f"False Negatives (Dynamic): {fn_pts.shape[0]}")
    
    # TNs are typically the bulk of the points if we consider GT static/unlabeled as negative class
    # And MDet non-dynamic (occluded, undetermined, empty) as negative prediction
    # For visualization, often these are just part of a general "other" or "background"
    # Let's make a "background" category from points not in RANSAC, TP, FP, FN
    
    already_categorized_mask = ransac_mask | tp_mask | fp_mask | fn_mask
    background_plot_mask = ~already_categorized_mask
    
    # Optionally, further categorize the background/TN if needed.
    # For now, all remaining points are background.
    background_pts_viz = raw_points_global_viz[background_plot_mask]
    if background_pts_viz.shape[0] > 0:
        k3d_points_to_plot['Background_Other'] = (
            background_pts_viz,
            k3d_cfg.get('tn_background_color_hex', 0xD3D3D3), # LightGray
            k3d_cfg.get('point_size', 0.05) * k3d_cfg.get('tn_background_point_size_factor', 0.5)
        )
    print(f"Background/Other points: {background_pts_viz.shape[0]}")

    # 5. Load GT Boxes for K3D
    if nusc and target_scene_rec_viz and TARGET_LIDAR_SD_TOKEN_VIZ and k3d_cfg.get('show_gt_boxes_k3d', True):
        try:
            sdd_rec = nusc.get('sample_data', TARGET_LIDAR_SD_TOKEN_VIZ)
            sample_token_for_boxes = sdd_rec['sample_token']
            all_scene_sweeps_list_for_boxes = list(get_scene_sweep_data_sequence(nusc, target_scene_rec_viz['token']))
            
            current_sweep_idx_for_boxes = -1
            for idx, sdd_box_loop in enumerate(all_scene_sweeps_list_for_boxes):
                if sdd_box_loop['lidar_sd_token'] == TARGET_LIDAR_SD_TOKEN_VIZ:
                    current_sweep_idx_for_boxes = idx
                    break
            
            if current_sweep_idx_for_boxes != -1:
                instance_tokens_in_scene_viz = find_instances_in_scene(nusc, target_scene_rec_viz['token'])
                for inst_token in instance_tokens_in_scene_viz:
                    boxes_for_inst, _, _, _ = get_interpolated_extrapolated_boxes_for_instance(
                        nusc, inst_token, all_scene_sweeps_list_for_boxes
                    )
                    box_at_sweep = boxes_for_inst[current_sweep_idx_for_boxes]
                    if box_at_sweep:
                        gt_boxes_for_k3d.append(box_at_sweep)
                print(f"Loaded {len(gt_boxes_for_k3d)} GT boxes for K3D plot.")
        except Exception as e_boxload:
            print(f"Error loading GT boxes for K3D: {e_boxload}")
else:
    print("Skipping K3D data preparation: Prerequisites not met.")

print("\nCell 3: K3D Data Preparation - OK")

In [ ]:
# %% Cell 4: K3D Visualization

if RUN_K3D and 'k3d_points_to_plot' in locals() and k3d_points_to_plot and config_accessor:
    k3d_cfg = config_accessor.get_k3d_plot_params()
    plot_title_k3d = f"Scene: {TARGET_SCENE_NAME_VIZ} - Sweep: {TARGET_LIDAR_SD_TOKEN_VIZ[:8]} - Eval"
    
    # Determine camera position (e.g., based on ego pose of the target sweep)
    initial_camera_k3d = None
    if target_sweep_data_dict_viz:
        T_global_lidar_viz = target_sweep_data_dict_viz['T_global_lidar']
        ego_center_viz = T_global_lidar_viz[:3, 3]
        initial_camera_k3d = [
            ego_center_viz[0] + k3d_cfg.get('camera_offset_x', 20), 
            ego_center_viz[1] + k3d_cfg.get('camera_offset_y', 20), 
            ego_center_viz[2] + k3d_cfg.get('camera_offset_z', 15),
            ego_center_viz[0], ego_center_viz[1], ego_center_viz[2], # Look at ego center
            0, 0, 1 # Up vector
        ]

    k3d_plot_eval = create_k3d_plot_with_points(
        points_dict=k3d_points_to_plot,
        plot_title=plot_title_k3d,
        background_color=k3d_cfg.get('plot_background_color_hex', 0xAAAAAA),
        grid_visible=k3d_cfg.get('grid_visible', True),
        camera_auto_fit=False if initial_camera_k3d else k3d_cfg.get('camera_auto_fit', True),
        initial_camera_settings=initial_camera_k3d
    )

    if k3d_plot_eval and 'gt_boxes_for_k3d' in locals() and gt_boxes_for_k3d and k3d_cfg.get('show_gt_boxes_k3d', True):
        print(f"  Adding {len(gt_boxes_for_k3d)} GT boxes to K3D plot...")
        num_instance_colors = k3d_cfg.get('num_instance_box_colors', 20)
        instance_colormap_name = k3d_cfg.get('instance_box_colormap', 'tab20')
        color_map_instances_k3d = plt.cm.get_cmap(instance_colormap_name, num_instance_colors)
        box_line_width = k3d_cfg.get('gt_box_line_width', 0.03)
        
        for i, box in enumerate(gt_boxes_for_k3d):
            instance_color_rgb_k3d = color_map_instances_k3d(i % num_instance_colors)[:3]
            instance_color_hex_k3d = int(instance_color_rgb_k3d[0]*255)<<16 | \
                                     int(instance_color_rgb_k3d[1]*255)<<8  | \
                                     int(instance_color_rgb_k3d[2]*255)
            corners = box.corners() # (3, 8)
            edges = [
                (0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4),
                (0, 4), (1, 5), (2, 6), (3, 7)
            ]
            for start_idx, end_idx in edges:
                segment = corners[:, [start_idx, end_idx]].T.astype(np.float32)
                k3d_plot_eval += k3d.line(segment, shader='simple', color=instance_color_hex_k3d, width=box_line_width,
                                          name=f'GTBox_{box.token[:6]}_{box.name[:5]}')
    
    if k3d_plot_eval:
        print("K3D plot object created, attempting to display...")
        display(k3d_plot_eval)
    else:
        print("Failed to create K3D plot.")
elif not RUN_K3D:
    print("K3D plotting is disabled (RUN_K3D=False).")
else:
    print("Skipping K3D Visualization: Prerequisites not met.")

print("\nCell 4: K3D Visualization - OK (if RUN_K3D was True)")